In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
# Needed imports

# Reading the silver layer delta table from product
silver_product = spark.table("accenture_final_project.silver_layer.product")

# Creating a surrogate key
windowSpec = Window.orderBy("ProductKey")

# We add the surrogate key and add the columns we want
dim_product = (
                silver_product 
                    .withColumn("ProductSK", row_number().over(windowSpec)) 
                    .select(
                        "ProductSK",
                        "ProductKey",
                        "Product",
                        "StandardCost",
                        "Color",
                        "Subcategory",
                        "Category",
                        "BackgroundColorFormat",
                        "FontColorFormat"
                    )
)

# Saving the dim table as a delta table in the gold layer
(
    dim_product
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("accenture_final_project.gold_layer.dim_product")
)
display(dim_product)